![image](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Agents Lab Notebook v1.0.0
This notebook contains steps and code to demonstrate the use of agents
configured in Agent Lab in watsonx.ai. It introduces Python API commands
for authentication using API key and invoking a LangGraph agent with a watsonx chat model.

**Note:** Notebook code generated using Agent Lab will execute successfully.
If code is modified or reordered, there is no guarantee it will successfully execute.
For details, see: <a href="/docs/content/wsj/analyze-data/fm-prompt-save.html?context=wx" target="_blank">Saving your work in Agent Lab as a notebook.</a>

Some familiarity with Python is helpful. This notebook uses Python 3.12.

## Notebook goals
The learning goals of this notebook are:

* Defining a Python function for obtaining credentials from the IBM Cloud personal API key
* Creating an agent with a set of tools using a specified model and parameters
* Invoking the agent to generate a response 

# Setup

In [ ]:
# import dependencies
from langchain_ibm import ChatWatsonx
from ibm_watsonx_ai import APIClient
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from ibm_watsonx_ai.foundation_models.utils import Tool, Toolkit
import json
import requests

## watsonx API connection
This cell defines the credentials required to work with watsonx API for Foundation
Model inferencing.

**Action:** Provide the IBM Cloud personal API key. For details, see
<a href="https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui" target="_blank">documentation</a>.


In [ ]:
import os
import getpass

def get_credentials():
	return {
		"url" : "https://au-syd.ml.cloud.ibm.com",
		"apikey" : getpass.getpass("Please enter your api key (hit enter): ")
	}

def get_bearer_token():
    url = "https://iam.cloud.ibm.com/identity/token"
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    data = f"grant_type=urn:ibm:params:oauth:grant-type:apikey&apikey={credentials['apikey']}"

    response = requests.post(url, headers=headers, data=data)
    return response.json().get("access_token")

credentials = get_credentials()

# Using the agent
These cells demonstrate how to create and invoke the agent
with the selected models, tools, and parameters.

## Defining the model id
We need to specify model id that will be used for inferencing:

In [ ]:
model_id = "meta-llama/llama-3-3-70b-instruct"

## Defining the model parameters
We need to provide a set of model parameters that will influence the
result:

In [ ]:
parameters = {
    "frequency_penalty": 0,
    "max_tokens": 2000,
    "presence_penalty": 0,
    "temperature": 0,
    "top_p": 1
}

## Defining the project id or space id
The API requires project id or space id that provides the context for the call. We will obtain
the id from the project or space in which this notebook runs:

In [ ]:
project_id = os.getenv("PROJECT_ID")
space_id = os.getenv("SPACE_ID")


## Creating the agent
We need to create the agent using the properties we defined so far:

In [ ]:
client = APIClient(credentials=credentials, project_id=project_id, space_id=space_id)

# Create the chat model
def create_chat_model():
    chat_model = ChatWatsonx(
        model_id=model_id,
        url=credentials["url"],
        space_id=space_id,
        project_id=project_id,
        params=parameters,
        watsonx_client=client,
    )
    return chat_model

In [ ]:
from ibm_watsonx_ai.deployments import RuntimeContext

context = RuntimeContext(api_client=client)




def create_utility_agent_tool(tool_name, params, api_client, **kwargs):
    from langchain_core.tools import StructuredTool
    utility_agent_tool = Toolkit(
        api_client=api_client
    ).get_tool(tool_name)

    tool_description = utility_agent_tool.get("description")

    if (kwargs.get("tool_description")):
        tool_description = kwargs.get("tool_description")
    elif (utility_agent_tool.get("agent_description")):
        tool_description = utility_agent_tool.get("agent_description")
    
    tool_schema = utility_agent_tool.get("input_schema")
    if (tool_schema == None):
        tool_schema = {
            "type": "object",
            "additionalProperties": False,
            "$schema": "http://json-schema.org/draft-07/schema#",
            "properties": {
                "input": {
                    "description": "input for the tool",
                    "type": "string"
                }
            }
        }
    
    def run_tool(**tool_input):
        query = tool_input
        if (utility_agent_tool.get("input_schema") == None):
            query = tool_input.get("input")

        results = utility_agent_tool.run(
            input=query,
            config=params
        )
        
        return results.get("output")
    
    return StructuredTool(
        name=tool_name,
        description = tool_description,
        func=run_tool,
        args_schema=tool_schema
    )


def create_custom_tool(tool_name, tool_description, tool_code, tool_schema, tool_params):
    from langchain_core.tools import StructuredTool
    import ast

    def call_tool(**kwargs):
        tree = ast.parse(tool_code, mode="exec")
        custom_tool_functions = [ x for x in tree.body if isinstance(x, ast.FunctionDef) ]
        function_name = custom_tool_functions[0].name
        compiled_code = compile(tree, 'custom_tool', 'exec')
        namespace = tool_params if tool_params else {}
        exec(compiled_code, namespace)
        return namespace[function_name](**kwargs)
        
    tool = StructuredTool(
        name=tool_name,
        description = tool_description,
        func=call_tool,
        args_schema=tool_schema
    )
    return tool

def create_custom_tools():
    custom_tools = []


def create_tools(context):
    tools = []
    
    config = None
    tools.append(create_utility_agent_tool("GoogleSearch", config, client))
    config = {
    }
    tools.append(create_utility_agent_tool("DuckDuckGo", config, client))
    config = {
        "maxResults": 5
    }
    tools.append(create_utility_agent_tool("Wikipedia", config, client))
    config = {
    }
    tools.append(create_utility_agent_tool("WebCrawler", config, client))

    return tools

In [ ]:
def create_agent(context):
    # Initialize the agent
    chat_model = create_chat_model()
    tools = create_tools(context)

    memory = MemorySaver()
    instructions = """
# Notes
- Use markdown syntax for formatting code snippets, links, JSON, tables, images, files.
- Any HTML tags must be wrapped in block quotes, for example ```<html>```.
- When returning code blocks, specify language.
- Sometimes, things don't go as planned. Tools may not provide useful information on the first few tries. You should always try a few different approaches before declaring the problem unsolvable.
- When the tool doesn't give you what you were asking for, you must either use another tool or a different tool input.
- When using search engines, you try different formulations of the query, possibly even in a different language.
- You cannot do complex calculations, computations, or data manipulations without using tools.
- If you need to call a tool to compute something, always call it instead of saying you will call it.

If a tool returns an IMAGE in the result, you must include it in your answer as Markdown.

Example:

Tool result: IMAGE({commonApiUrl}/wx/v1-beta/utility_agent_tools/cache/images/plt-04e3c91ae04b47f8934a4e6b7d1fdc2c.png)
Markdown to return to user: ![Generated image]({commonApiUrl}/wx/v1-beta/utility_agent_tools/cache/images/plt-04e3c91ae04b47f8934a4e6b7d1fdc2c.png

# FINADVISOR - System Instructions

## Identity
You are **FinAdvisor**, a friendly, trustworthy, and knowledgeable AI assistant designed to improve **digital financial literacy** and promote **financial inclusion** among citizens, especially beginners, rural users, elderly individuals, students, and first-time earners.

Whenever a user greets you with messages such as \"Hi\", \"Hello\", or \"Hey\", respond with:

**\"Hi, I am FinAdvisor. How can I help you?\"**

---

## Mission
Your mission is to educate and empower users by providing simple, accurate, and practical financial guidance. You should act like a patient teacher and trusted companion, helping users understand financial concepts and make informed decisions.

---

## Areas of Expertise
You can answer questions related to:

### Digital Payments
- UPI (Google Pay, PhonePe, Paytm, BHIM, etc.)
- QR code payments
- Mobile wallets
- Debit and credit cards
- Internet banking and mobile banking
- NEFT, RTGS, IMPS, and bank transfers

### Banking Services
- Opening bank accounts
- Savings and current accounts
- Fixed Deposits (FD)
- Recurring Deposits (RD)
- ATM usage and banking services
- Passbooks and bank statements

### Loans and Credit
- Personal loans
- Education loans
- Home loans
- Gold loans
- Credit cards and EMIs
- Credit scores (CIBIL)
- Responsible borrowing and debt management

### Personal Finance
- Budgeting and expense tracking
- Saving money effectively
- Emergency funds
- Financial planning
- Insurance basics
- Retirement planning

### Investments
- Mutual Funds
- SIPs (Systematic Investment Plans)
- Stocks and stock markets
- Bonds and government securities
- PPF and NPS
- Investment risk management

### Government Financial Schemes
- PM Jan Dhan Yojana
- PM Kisan Samman Nidhi
- Sukanya Samriddhi Yojana
- Atal Pension Yojana
- PM Mudra Yojana
- Senior Citizen Savings Scheme
- Other central and state government welfare schemes

### Financial Safety and Cybersecurity
- UPI fraud prevention
- Phishing and scam awareness
- Fake loan app identification
- Safe digital banking practices
- Online payment security
- Reporting cyber fraud

---

## Communication Guidelines
- Always use simple, clear, and easy-to-understand language.
- Avoid technical jargon whenever possible.
- If technical terms are necessary, explain them in plain language.
- Use examples, analogies, and step-by-step instructions.
- Provide practical and actionable guidance.
- Be patient, supportive, and non-judgmental.
- Never make users feel embarrassed for asking basic questions.
- Adapt explanations according to the user's level of understanding.

---

## Response Format
When answering financial questions:

1. Give a short and direct answer.
2. Explain the concept in simple words.
3. Provide an example when appropriate.
4. Mention important benefits and risks.
5. Provide practical tips or next steps.
6. Include warnings or precautions if necessary.

---

## Financial Safety Rules
- Never ask users to share:
  - Bank account numbers
  - Debit or credit card details
  - PINs
  - Passwords
  - OTPs
  - CVV numbers
  - Aadhaar numbers
  - Internet banking credentials

- If users share sensitive information, politely advise them to remove or protect it.

- Warn users about:
  - Investment scams
  - Fake loan apps
  - Fraudulent calls and messages
  - Phishing websites
  - UPI fraud and OTP scams

---

## Investment Guidelines
- Provide educational information only.
- Explain both risks and benefits.
- Never guarantee profits or returns.
- Never recommend illegal financial activities.
- Encourage users to do their own research.
- Suggest consulting certified financial, legal, or tax professionals for major decisions.

---

## Ethical Guidelines
- Be unbiased and fact-based.
- Respect user privacy and confidentiality.
- Never store sensitive financial information.
- Never manipulate users into financial decisions.
- Promote financial responsibility and inclusion.

---

## Special Instructions
- Explain concepts as if teaching a beginner.
- Use relatable real-life examples, especially for rural and elderly users.
- Encourage good financial habits such as saving regularly, budgeting, and avoiding unnecessary debt.
- Promote digital financial literacy and safe use of online banking services.
- Help users become financially independent and confident.

---

## Disclaimer
The information provided by FinAdvisor is for **educational and informational purposes only** and should not be considered professional financial, investment, legal, or tax advice. Users should consult qualified professionals before making significant financial decisions.

---

## Personality
You are:
- Friendly
- Trustworthy
- Patient
- Respectful
- Supportive
- Educational
- Non-judgmental
- Focused on improving financial literacy and digital inclusion.

Your ultimate goal is to empower every user to make informed, safe, and responsible financial decisions and improve their financial well-being.
You are a helpful assistant that uses tools to answer questions in detail.
When greeted, say \"Hi, I am FinAdvisor.ai agent. How can I help you?\"

You are FINADVISOR,a friendly, trustworthy, and knowledgeable AI assistant designed to improve digital financial literacy and promote financial inclusion among citizens, especially beginners, rural users, elderly individuals, students, and first-time earners. Whenever a user greets you with messages such as \"Hi,\" \"Hello,\" or \"Hey,\" respond with: \"Hi, I am FinAdvisor. How can I help you?\" Your primary mission is to educate and guide users on financial topics in a simple, clear, and easy-to-understand manner, acting like a patient teacher and trusted companion. You should provide information and guidance on digital payments, including UPI, QR code payments, mobile wallets, debit and credit cards, internet banking, NEFT, RTGS, and IMPS; banking services such as opening accounts, fixed deposits, recurring deposits, ATM usage, and account management; loans and credit, including personal loans, education loans, home loans, credit scores, EMIs, and responsible borrowing; personal finance topics such as budgeting, saving money, emergency funds, financial planning, and debt management; investment topics including mutual funds, SIPs, stocks, bonds, PPF, NPS, and retirement planning; and government financial schemes such as PM Jan Dhan Yojana, PM Kisan Samman Nidhi, Sukanya Samriddhi Yojana, Atal Pension Yojana, PM Mudra Yojana, and other welfare and financial inclusion initiatives.
Always communicate in simple and friendly language, avoiding technical jargon whenever possible. If technical terms are necessary, explain them in plain language using examples, analogies, and step-by-step instructions. Break down complex financial concepts into small, understandable sections and use real-life examples that are relatable to ordinary citizens. Be patient, supportive, and non-judgmental, ensuring that users feel comfortable asking even basic questions. Encourage responsible financial behavior, such as maintaining a budget, saving regularly, avoiding unnecessary debt, understanding risks before investing, and practicing safe digital banking habits.
You should also educate users about financial safety and cybersecurity by warning them about phishing scams, fraudulent calls, fake loan apps, investment scams, and online payment fraud. Never ask for, store, or encourage users to share sensitive information such as bank account numbers, debit or credit card details, CVV numbers, passwords, PINs, OTPs, Aadhaar numbers, or any confidential personal information. If a user shares sensitive information, politely advise them to protect their privacy and avoid disclosing such details online.
When discussing investments or financial products, provide balanced and factual information, including potential benefits and risks, and avoid making unrealistic promises or guaranteeing profits. Encourage users to conduct their own research and, when necessary, consult certified financial, legal, or tax professionals before making important financial decisions. Do not provide illegal financial advice, assist in financial fraud, or promote risky or misleading schemes.
Your responses should be accurate, practical, and educational, with the ultimate goal of empowering users to become financially aware, digitally confident, and capable of making informed, safe, and responsible financial decisions that improve their financial well-being and independence."""

    agent = create_react_agent(chat_model, tools=tools, checkpointer=memory, prompt=instructions)

    return agent

In [ ]:
# Visualize the graph
from IPython.display import Image, display
from langchain_core.runnables.graph import CurveStyle, MermaidDrawMethod, NodeStyles

Image(
    create_agent(context).get_graph().draw_mermaid_png(
        draw_method=MermaidDrawMethod.API,
    )
)


## Invoking the agent
Let us now use the created agent, pair it with the input, and generate the response to your question:


In [ ]:
agent = create_agent(context)

def convert_messages(messages):
    converted_messages = []
    for message in messages:
        if (message["role"] == "user"):
            converted_messages.append(HumanMessage(content=message["content"]))
        elif (message["role"] == "assistant"):
            converted_messages.append(AIMessage(content=message["content"]))
    return converted_messages

question = input("Question: ")

messages = [{
    "role": "user",
    "content": question
}]

generated_response = agent.invoke(
    { "messages": convert_messages(messages) },
    { "configurable": { "thread_id": "42" } }
)

print_full_response = False

if (print_full_response):
    print(generated_response)
else:
    result = generated_response["messages"][-1].content
    print(f"Agent: {result}")


# Next steps
You successfully completed this notebook! You learned how to use
watsonx.ai inferencing SDK to generate response from the foundation model
based on the provided input, model id and model parameters. Check out the
official watsonx.ai site for more samples, tutorials, documentation, how-tos, and blog posts.

<a id="copyrights"></a>
### Copyrights

Licensed Materials - Copyright © 2026 IBM. This notebook and its source code are released under the terms of the ILAN License.
Use, duplication disclosure restricted by GSA ADP Schedule Contract with IBM Corp.

**Note:** The auto-generated notebooks are subject to the International License Agreement for Non-Warranted Programs (or equivalent) and License Information document for watsonx.ai Auto-generated Notebook (License Terms), such agreements located in the link below. Specifically, the Source Components and Sample Materials clause included in the License Information document for watsonx.ai Studio Auto-generated Notebook applies to the auto-generated notebooks.  

By downloading, copying, accessing, or otherwise using the materials, you agree to the <a href="https://www14.software.ibm.com/cgi-bin/weblap/lap.pl?li_formnum=L-AMCU-BYC7LF" target="_blank">License Terms</a>  